# Average Out Degree of Nearest Neighbors
## Source: out, Target: out

### Imports

In [1]:
from __future__ import annotations
import networkx as nx
import numpy as np
import pytest
from abc import ABC, abstractmethod

### Error Classes

In [2]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass

class EmptyGraphError(Exception):
    """Exception raised for empty graph. Nodes with no edges."""
    pass

def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

### Abstract Class

In [3]:
class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no Average Out Degree.")

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass

### Decorators

In [4]:
def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls

### Auxiliar Fxns

In [5]:
def remove_self_loops(G: nx.DiGraph, n_nodes):
    G.remove_edges_from([(i,i) for i in range(n_nodes)])
    return G

def get_parent_nodes(G: nx.DiGraph):
    """Get the parent nodes of a graph."""
    return [i for i, k_out in G.out_degree() if k_out > 0]

### Average Out Degree of Nearest Neighbors Class
In this case, the neighborhood selected for each node is it's list of successors. The calculation is the average of each neighbor's out degree. 
The biological normalization excludes all the 0s that come from nodes that do not even have an out-degree higher than 0.

In [17]:
@return_distribution
@use_direction
class AverageOutDegreeNearestNeighbors(_Property): # Hereda de la clase _Property
    """Average Out-Degree of Nearest Neighbors

    Average out-degree of nearest neighbors is defined as de average of out-degrees of each memeber of a node's neighborhood.
    In this case, the neighborhood of each node is the list of its successors.

    Methods:
        compute: Compute the average out-degree for each node in the graph.
        norm_biol: Normalize the average out-degree of nearest neighbors to all nodes in the graph AND eliminate uninformative 0s.
        norm_network: Normalize the average out-degree of nearest neighbors to all nodes in the graph.
    """
    __name__ = 'Average Degree for Nearest Neighbors (Out-Out)'

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)

    def compute(self) -> np.array:
        """Compute the average out-degree of nearest neighbors using Networkx implementation.

        Returns:
            np.array: average out-degree for each node in the graph.
        """
        self.A = nx.DiGraph
        self.A = remove_self_loops(self.G, self._n_nodes)
        self._dict_av_degree = nx.average_neighbor_degree(self.A, source='out', target='out')
        self._raw_value = np.fromiter(self._dict_av_degree.values(), dtype=float)

        return self._raw_value

    @check_raw_value    # Decorator to check if raw value is None. If it is, raise an error.
    def norm_biol(self) -> np.array:
        """Normalize the average degree of nearest neighbors for every node in the graph to the number of nodes and exclude
            0s from nodes that do not have a out-degree higher than 0. Relation between order of values and order of nodes is lost.
        """
        parents_value = np.array([self._dict_av_degree[node] for node in get_parent_nodes(self.A)])
        return parents_value * (1 / (self._n_nodes - 1))

    @check_raw_value
    def norm_network(self) -> np.array:
        """Normalize the average degree for nearest neighbors to the number of nodes (-1 because you can not be your own neighbor)"""
        return self._raw_value * (1 / (self._n_nodes - 1))


### Testing

In [20]:
# Null graph
B = nx.DiGraph()

with pytest.raises(NullGraphError) as e_info:
    property = AverageOutDegreeNearestNeighbors(B)

# Empty graph, allows instance from an empty graph, but does not compute
n_nodes= 5
B.add_nodes_from(range(n_nodes))
property = AverageOutDegreeNearestNeighbors(B)

assert not np.any(property.compute()) #All 0s

# add edges
# complete directed graph with self loops
B.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = AverageOutDegreeNearestNeighbors(B)
expected = np.array([4., 4., 4., 4., 4.])
expected_1 = np.array([1., 1., 1., 1., 1.])

assert np.allclose(property.compute(), expected)
assert np.allclose(property.norm_biol(), expected_1)
assert np.allclose(property.norm_network(), expected_1)

# add edges
# only half of the nodes are parents and regulate every node in the graph
B = nx.DiGraph()
n_nodes= 5
B.add_nodes_from(range(n_nodes))
B.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = AverageOutDegreeNearestNeighbors(B)
expected = np.array([1., 1., 0., 0., 0.])
expected_1 = np.array([0.25, 0.25])
expected_2 = np.array([0.25, 0.25, 0.  , 0.  , 0.  ])

assert np.allclose(property.compute(), expected)
assert np.allclose(property.norm_biol(), expected_1)
assert np.allclose(property.norm_network(), expected_2)
